In [7]:
import os
import shutil
import kagglehub
from kagglehub import KaggleDatasetAdapter

# -----------------------------
# Load metadata CSV
# -----------------------------
df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "kmader/skin-cancer-mnist-ham10000",
    "HAM10000_metadata.csv"
)
print("First 5 records:\n", df.head())

# -----------------------------
# Set paths to image folders
# -----------------------------
part1_path = "/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_1"
part2_path = "/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_2"

TARGET_DIR = '/kaggle/working/HAM10000_split'

df['image_id'] = df['image_id'] + '.jpg'

# Create class folders
classes = df['dx'].unique()
for cls in classes:
    os.makedirs(os.path.join(TARGET_DIR, 'all_data', cls), exist_ok=True)

# Copy images from both parts
for _, row in df.iterrows():
    fname = row['image_id']
    # Check in part 1
    if os.path.exists(os.path.join(part1_path, fname)):
        src = os.path.join(part1_path, fname)
    # Check in part 2
    elif os.path.exists(os.path.join(part2_path, fname)):
        src = os.path.join(part2_path, fname)
    else:
        print(f"WARNING: {fname} not found in any folder!")
        continue

    dst = os.path.join(TARGET_DIR, 'all_data', row['dx'], fname)
    shutil.copyfile(src, dst)


First 5 records:
      lesion_id      image_id   dx dx_type   age   sex localization
0  HAM_0000118  ISIC_0027419  bkl   histo  80.0  male        scalp
1  HAM_0000118  ISIC_0025030  bkl   histo  80.0  male        scalp
2  HAM_0002730  ISIC_0026769  bkl   histo  80.0  male        scalp
3  HAM_0002730  ISIC_0025661  bkl   histo  80.0  male        scalp
4  HAM_0001466  ISIC_0031633  bkl   histo  75.0  male          ear


In [8]:
# Block 2: Train/Validation Split
import os
from sklearn.model_selection import train_test_split

for cls in classes:
    files = os.listdir(os.path.join(TARGET_DIR, 'all_data', cls))
    train_files, val_files = train_test_split(files, test_size=0.2, random_state=42)
    
    # Create folders
    os.makedirs(os.path.join(TARGET_DIR, 'train', cls), exist_ok=True)
    os.makedirs(os.path.join(TARGET_DIR, 'val', cls), exist_ok=True)
    
    # Move files to train
    for f in train_files:
        src = os.path.join(TARGET_DIR, 'all_data', cls, f)
        dst = os.path.join(TARGET_DIR, 'train', cls, f)
        os.rename(src, dst)
    
    # Move files to val
    for f in val_files:
        src = os.path.join(TARGET_DIR, 'all_data', cls, f)
        dst = os.path.join(TARGET_DIR, 'val', cls, f)
        os.rename(src, dst)


In [9]:
# Block 3: PyTorch Dataloaders
import torch
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

# Augmentations for training
train_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

train_dataset = ImageFolder(os.path.join(TARGET_DIR, 'train'), transform=train_transforms)
val_dataset = ImageFolder(os.path.join(TARGET_DIR, 'val'), transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)


In [10]:
# Block 4: Class Weights
import numpy as np
import torch.nn as nn

# Count samples per class
class_counts = [0]*len(classes)
for _, label in train_dataset.samples:
    class_counts[label] += 1

# Compute class weights
class_weights = [sum(class_counts)/c for c in class_counts]
class_weights = torch.tensor(class_weights, dtype=torch.float).cuda()

# CrossEntropyLoss with class weights
criterion = nn.CrossEntropyLoss(weight=class_weights)


In [18]:
import timm
import torch

num_classes = 7  # HAM10000 classes

# Create Swin Transformer with correct classification head
model = timm.create_model(
    'swin_base_patch4_window7_224',
    pretrained=True,
    num_classes=num_classes  # ← this automatically replaces head correctly
)

model = model.cuda()


In [19]:
# Freeze backbone, train only head
for param in model.parameters():
    param.requires_grad = False
for param in model.head.parameters():
    param.requires_grad = True

import torch.optim as optim
from tqdm.notebook import tqdm

optimizer = optim.Adam(model.head.parameters(), lr=1e-3)

def train_one_epoch(loader, model, criterion, optimizer):
    model.train()
    running_loss = 0
    for imgs, labels in tqdm(loader, total=len(loader)):
        imgs = imgs.cuda(non_blocking=True)
        labels = labels.cuda(non_blocking=True).long()
        
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    return running_loss / len(loader)

# Train head for 5 epochs
for epoch in range(5):
    loss = train_one_epoch(train_loader, model, criterion, optimizer)
    print(f"Phase 1 - Epoch {epoch+1}/5, Loss: {loss:.4f}")


  0%|          | 0/251 [00:00<?, ?it/s]

Phase 1 - Epoch 1/5, Loss: 1.3630


  0%|          | 0/251 [00:00<?, ?it/s]

Phase 1 - Epoch 2/5, Loss: 1.0644


  0%|          | 0/251 [00:00<?, ?it/s]

Phase 1 - Epoch 3/5, Loss: 0.9886


  0%|          | 0/251 [00:00<?, ?it/s]

Phase 1 - Epoch 4/5, Loss: 0.9270


  0%|          | 0/251 [00:00<?, ?it/s]

Phase 1 - Epoch 5/5, Loss: 0.8812


In [20]:
# Unfreeze all layers
for param in model.parameters():
    param.requires_grad = True

# Optimizer and scheduler for full model fine-tuning
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)

best_val_acc = 0
import copy

for epoch in range(15):
    # -----------------------
    # Training
    # -----------------------
    model.train()
    running_loss = 0
    correct_train = 0
    total_train = 0

    for imgs, labels in tqdm(train_loader, total=len(train_loader)):
        imgs, labels = imgs.cuda(), labels.cuda()
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

        # Training accuracy
        _, preds = torch.max(outputs, 1)
        correct_train += (preds == labels).sum().item()
        total_train += labels.size(0)
    
    train_acc = correct_train / total_train
    scheduler.step()
    
    # -----------------------
    # Validation
    # -----------------------
    model.eval()
    correct_val = 0
    total_val = 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.cuda(), labels.cuda()
            outputs = model(imgs)
            _, preds = torch.max(outputs, 1)
            correct_val += (preds == labels).sum().item()
            total_val += labels.size(0)
    
    val_acc = correct_val / total_val

    print(f"Phase 2 - Epoch {epoch+1}/15, "
          f"Train Loss: {running_loss/len(train_loader):.4f}, "
          f"Train Acc: {train_acc:.4f}, "
          f"Val Acc: {val_acc:.4f}")
    
    # Save best model weights
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_wts = copy.deepcopy(model.state_dict())


  0%|          | 0/251 [00:00<?, ?it/s]

Phase 2 - Epoch 1/15, Train Loss: 1.0229, Train Acc: 0.6472, Val Acc: 0.6813


  0%|          | 0/251 [00:00<?, ?it/s]

Phase 2 - Epoch 2/15, Train Loss: 0.6728, Train Acc: 0.7258, Val Acc: 0.7416


  0%|          | 0/251 [00:00<?, ?it/s]

...

Phase 2 - Epoch 13/15, Train Loss: 0.0272, Train Acc: 0.9775, Val Acc: 0.9077


  0%|          | 0/251 [00:00<?, ?it/s]

Phase 2 - Epoch 14/15, Train Loss: 0.0227, Train Acc: 0.9806, Val Acc: 0.9047


  0%|          | 0/251 [00:00<?, ?it/s]

Phase 2 - Epoch 15/15, Train Loss: 0.0188, Train Acc: 0.9831, Val Acc: 0.9052


In [21]:
from sklearn.metrics import classification_report, confusion_matrix
import torch

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for imgs, labels in val_loader:
        imgs, labels = imgs.cuda(), labels.cuda()
        outputs = model(imgs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
print("Confusion Matrix:\n", cm)

# Classification Report
report = classification_report(all_labels, all_preds, target_names=classes)
print("Classification Report:\n", report)


Confusion Matrix:
 [[  50    2    7    0    6    1    0]
 [   3   91    3    2    1    3    0]
 [   9    2  181    0   16   12    0]
 [   0    1    0   22    0    0    0]
 [   1    0    3    0  175   44    0]
 [   3    5   16    0   45 1270    2]
 [   0    1    0    0    1    1   26]]
Classification Report:
               precision    recall  f1-score   support

         bkl       0.76      0.76      0.76        66
          nv       0.89      0.88      0.89       103
          df       0.86      0.82      0.84       220
         mel       0.92      0.96      0.94        23
        vasc       0.72      0.78      0.75       223
         bcc       0.95      0.95      0.95      1341
       akiec       0.93      0.90      0.91        29

    accuracy                           0.91      2005
   macro avg       0.86      0.86      0.86      2005
weighted avg       0.91      0.91      0.91      2005



In [22]:
# Save full model state_dict
save_path = '/kaggle/working/SwinTransformer_best.pth'
torch.save(model.state_dict(), save_path)
print(f"Model saved successfully at: {save_path}")

# Optional: Save the entire model (including architecture) for easy loading
save_full_model_path = '/kaggle/working/SwinTransformer_full_model.pth'
torch.save(model, save_full_model_path)
print(f"Full model (architecture + weights) saved at: {save_full_model_path}")


Model saved successfully at: /kaggle/working/SwinTransformer_best.pth
Full model (architecture + weights) saved at: /kaggle/working/SwinTransformer_full_model.pth


In [ ]:
# ============================================================
# ✅ TEST — PREDICT LESION TYPE USING TRAINED SWIN TRANSFORMER MODEL
# ============================================================

import torch
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
from google.colab import files
import timm

# -----------------------------
# Load the saved model
# -----------------------------
model_path = "/kaggle/working/SwinTransformer_best.pth"  # replace with your path
classes = [
    "akiec (Actinic Keratoses / Intraepithelial Carcinoma)",
    "bcc (Basal Cell Carcinoma)",
    "bkl (Benign Keratosis-like Lesions)",
    "df (Dermatofibroma)",
    "mel (Melanoma)",
    "nv (Melanocytic Nevi)",
    "vasc (Vascular Lesions)"
]

# Recreate the model architecture
model = timm.create_model('swin_base_patch4_window7_224', pretrained=False, num_classes=len(classes))
model.load_state_dict(torch.load(model_path))
model.eval()
model = model.cuda()

# -----------------------------
# Define preprocessing
# -----------------------------
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],  # ImageNet stats
                         std=[0.229, 0.224, 0.225])
])

# -----------------------------
# Upload image
# -----------------------------
uploaded = files.upload()

for fn in uploaded.keys():
    img_path = fn
    img = Image.open(img_path).convert("RGB")
    input_tensor = preprocess(img).unsqueeze(0).cuda()  # Add batch dimension

    # -----------------------------
    # Predict
    # -----------------------------
    with torch.no_grad():
        outputs = model(input_tensor)
        probs = torch.softmax(outputs, dim=1)
        confidence, pred_idx = torch.max(probs, 1)
    
    # -----------------------------
    # Show results
    # -----------------------------
    plt.imshow(img)
    plt.axis("off")
    plt.title(f"Predicted: {classes[pred_idx.item()]}\nConfidence: {confidence.item()*100:.2f}%")
    plt.show()

    print("\n✅ Prediction:")
    for i, label in enumerate(classes):
        print(f"{label:55s} -> {probs[0, i].item()*100:.2f}%")
